# Liquid Neural Networks: A Novel Approach to Dynamic Neural Computation

By Sydney Bach (design / architecture), Claude.ai (math markup, code).

## North-star status

This notebook is the proof-of-concept for the **Liquid + Transformer hybrid
block** that sits at the heart of SolaceCore's `InferenceCube` design. It is
preserved here as a living design document. Where the original Kaggle proof
diverged from the math, this revision pulls them back into alignment.

The system thesis is:

* A **block** owns a chunk of inference (a "cube").
* While the LTC inside the block is still learning, the cube is computed by
  the **transformer path** (multi-head attention output is the cube's content,
  which is a vector).
* When the LTC's per-cube error drops below threshold, ownership of the cube
  flips from `TRANSFORMER` to `LNN_OWNED`. The cube is now produced by a
  single LTC cell's continuous-time dynamics. The cube has gone **from a
  vector to a neuron**.
* Post-takeover, the LTC keeps learning from input but is no longer compared
  against the transformer's output. It is the local truth for that cube. This
  is plasticity after takeover.

The Kotlin port maps this to actors:
* One block = one Actor wrapping a `TransformerLNN`.
* Inputs / outputs = typed Ports. The "virtual dendrites" and "cortical
  connections" of the design.
* The phonological loop = a back-edge port from a block's output, through
  Reflection Memory, back into one of its inputs (or an upstream block's).
* The neural tree = the actor tree under coroutine parentage.

## Mathematical Framework

### 1. Continuous-time LTC dynamics

The Liquid Time-Constant cell follows the closed-form CfC formulation
(Hasani et al., 2022) with the per-neuron time constant explicit so the
bounded-dynamics theorem from Hasani et al. (2020) still holds in spirit.

Per timestep:

$$
z_t = \big[\,x_t \;;\; h_{t-1}\,\big]
$$

$$
\text{features} = \text{backbone}(z_t)
$$

$$
f_t = \sigma\bigl(\text{time\_net}(\text{features})\bigr)
$$

$$
g_x = \text{state\_net\_g}(\text{features})
\qquad
h_x = \text{state\_net\_h}(\text{features})
$$

$$
\text{gate} = \sigma\!\left(-\,\frac{f_t}{\tau}\,t\right)
$$

$$
h_t = \text{gate}\,\odot\,g_x \;+\; (1-\text{gate})\,\odot\,(h_x + A)
$$

Where:
* $\tau = \operatorname{softplus}(\tau_{\text{raw}}) + \epsilon$ is a
  positive per-neuron time constant. Larger $\tau$ slows the gate's
  transition from the short-term branch to the long-term branch.
* $A$ is the input-driven equilibrium offset present on the long-term
  branch.
* $f_t$ is the per-neuron transition rate produced by `time_net`.
* $g_x, h_x$ are the short-term and long-term state transformations.

### 2. Bounded dynamics

Following Hasani et al. (2020):

$$
\frac{\tau_i}{1 + \tau_i\,W_i} \,\le\, \tau_{\text{sys}_i} \,\le\, \tau_i
$$

The implementation keeps $\tau$ positive via softplus and the gate as a
sigmoid, so the system's effective time constants stay bounded under
arbitrary $W_i$.

## Architecture: Transformer + LTC hybrid block

Inside one block, the data flow is **serial composition with a residual
safety wire**, not two parallel paths fused at the end:

```
x ──→ input_proj ──→ h
                    │
                    ├──→ self-attention(Q=K=V=h) ──→ h_att
                    │     h_att = norm1(h + h_att)         (residual)
                    │
                    └──→ for t in 0..L:
                            ltc_state ← ltc(h_att[:,t], ltc_state, times[:,t])
                            outputs.append(ltc_state)

                         outputs = stack(outputs)
                         outputs = norm2(outputs + h_att)  (residual)
                         y       = output_proj(outputs)
```

Attention prepares globally-mixed content; the LTC walks it forward under
time. The residual is what preserves the attention signal in case the LTC
washes it out during training.

## Cube ownership state machine (target)

The block above is what runs when a cube is `TRANSFORMER`-owned or
`MENTORING`. The InferenceCube design adds a state machine the prototype
does not yet implement:

| State          | Meaning                                                                  |
|----------------|--------------------------------------------------------------------------|
| `TRANSFORMER`  | Cube content = attention output. LTC observes but does not contribute.   |
| `MENTORING`    | Cube content = attention output. LTC trains against attention output.    |
| `LNN_OWNED`    | LTC error < ε. Cube content = LTC dynamic state. Attention is bypassed.  |
| `FROZEN`       | Lobe corresponds to an old transformer version. Read-only, auxiliary.   |

The transition `MENTORING → LNN_OWNED` is the vector-to-neuron event for
that cube.

## Memory signature

A consequence of having an LTC inside every block is that the cell's
hidden state at any moment is a **time-integrated affective fingerprint**
of everything the block has seen. In SolaceCore, that hidden state is the
signature stamped on each Reflection Memory entry the block writes.

Memory retrieval becomes:

> Find entries whose stored LTC state correlates with the current LTC
> state, within a freshness-decay window.

This is the "rhyming" mechanism: it captures affective resonance rather
than surface lexical similarity, and it does so without any separate
spike-train kernel — the liquid layer is already producing the right
fingerprint.

## What this notebook proves

* The closed-form LTC cell trains end-to-end through a multi-head
  attention layer under MSE on synthetic multi-frequency time series.
* The serial attention → LTC → residual fusion is stable.
* Per-neuron time gating (after the τ/A correction below) is a working
  blender between short-term and long-term branches.

## What this notebook does not yet implement

* Cube partitioning of `seq_len` into independent sub-blocks with their
  own LTC state.
* Cube-to-cube communication, $C_{ij} = f(W_i C_i + W_j C_j)$.
* The mentoring / ownership state machine (currently joint training).
* Per-cube error tracking and threshold-based ownership flip.
* Lobe versioning (freezing on transformer update; growing new lobes).
* Dream replay biased by signature density during low load.
* Block-to-block topology — this is one block; the SolaceCore vision is a
  graph of blocks connected via Kotlin actors and typed ports.
* Phonological loop — recurrent rehearsal back-edge.

These are the engineering items the Kotlin port adds on top of the
proven primitive.

## Implementation guidelines (for the Kotlin port)

1. Keep $\tau$ positive (softplus + epsilon) for stability.
2. Use Xavier / spectral-norm-bounded weight init so $\|W_i\|$ stays small
   relative to $\sqrt{2/n}$.
3. The `for t in range(seq_len)` LTC loop is **serial in time** by design.
   Across cubes within a block it can be parallel; across batches it is
   parallel. Across timesteps within a cube it is causal and serial.
4. Treat `LTC.state` after a forward pass as the canonical Reflection
   Memory signature (hidden_size float vector).
5. Ownership flips are atomic per-cube; a block holds an array of cube
   ownership flags, not a single global flag.
6. Frozen lobes contribute as auxiliary inputs to new lobes — never delete
   trained behavior on transformer update.
7. For the Kotlin port, train in a PyTorch sidecar process and run frozen
   inference in pure Kotlin (e.g., via ONNX runtime) once cubes hit
   `LNN_OWNED`. The Kotlin actor system owns the topology and the
   ownership state machine; PyTorch owns the gradients.


## Implementation walkthrough

The hybrid is two well-known components composed in series with a residual
safety wire:

* `nn.MultiheadAttention` — bidirectional, parallel, content-mixing across
  the whole sequence at once.
* `LiquidTimeConstant` — unidirectional, serial in time, causal continuous-
  time integration.

```python
class TransformerLNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_heads=4):
        # Transformer path (parallel content mixing)
        self.attention = nn.MultiheadAttention(...)
        # LTC path (serial time integration, post-attention)
        self.ltc       = LiquidTimeConstant(hidden_size, hidden_size)
```

### Data generation (synthetic)

The training data is a stack of sine waves at multiple frequencies, with a
shifted/scaled target so the model has to learn both temporal dependency
and amplitude transformation:

```python
for i in range(input_dim):
    freq1, freq2 = (i + 1) * 0.5, (i + 1) * 0.25
    pattern = sin(2*pi*freq1*t) + 0.5 * sin(2*pi*freq2*t)
```

This is a placeholder substrate. Real deployment substitutes Reflection
Memory entries (with LTC-state signatures) as the input stream and uses
the upstream LLM's output as the mentoring target during the
`MENTORING` ownership phase.

### Training loop

The Kaggle prototype trains attention + LTC + projections jointly under
MSE. That is fine for proving the hybrid composes; it is not the
InferenceCube training regime. The InferenceCube regime trains the LTC
*to mirror the transformer's output for a specific cube* and tracks per-
cube error so it can decide when to flip ownership. The Kotlin port adds
that ownership state machine; this notebook does not.


# The key innovation here is how it combines:


* Transformer's parallel processing (attention)
* LTC's continuous time dynamics (liquid state)
* Time-dependent gating to balance both

# Overview: Transformer-LNN hybrid approach
[![](https://mermaid.ink/img/pako:eNqNlftu2jAUxl_FcqWJdqEihNsidRKBtnSjgEomoZEqMolDMoKdJY6AXv7dA-wR9ySzHS4Ja7tGCMXh-53z-eT48Agd6mKoQy-kK8dHMQNm1yKAX0k6m8co8sENiVJmYpLQOJlaUC7Bdm3B-0wtrgn_dQ2mM8QcXwEJ_mmHmCggEICdBA_4viA3uZwFS5wcIwcZJq5FjuyMYvoDOyygZG_m8KjoB5TLn8HobviFK_sBwSguHcyAP79-Az9wXUzk-rTACkriPc76_24qB75tt80YJsKaEVJnwYPtHwjbDk6SgMwLqXsyr3nXHoxVUaIYkSSiCS5VFLVoMtNkNtumqKdvI8bAdG9zaztzWyy_ICSakbdpyIIeRu7eX0G9E8uU1YItVam8YKsq5YPh3a3YRB9tcDyg8fLNWvXNzq5K_Pa1-siYMjoX9YfDEZdf0Rhg5PhANhTDEWB6AdpKJdYZDjptsWcHsdJ0bTNeIJvdK8ANlhfloxpnYskZBmcM5CxmlGAwwGxF40VBbBhSOB71b0T8cRQGDDAKNBAh5hdPixRJ-ZMFTW5bRLTgE7gSqGczcAGSYL6kgVsSu7IJZqevRxgzxDC4FgGuxUGc22sRQDwVpD3_H9oTaG8im6iI-gX0ygQfQMZet81LkUrgB7Nl4f0MHJkVWg5eT_hXLzuZRv9y0JX5CF7xADLOGRDOP4KSWhbrU_6A-3mzbYYp44d61znZ6rXmkTmzl2S2O1-5nu_TWZSopJKsB45aQCr37Vx9sZ13jZm1_fCbWRw7uXEh585hDBVTbcFdEB5g876BmquLE6Ik6WIPMDmjOzTkh8MLwlA_8T55SsJiusD6iaZp2_vyKnCZr1ej9VEAGuXh2ew9cC4EmCg9RYwZhW8l7yavEdtVhMYwFNEjSvaGtqmhApc4XqLA5f9Rj4Ljk8fHS2xBnd-62EN8blnQIs9cilJGxxviQJ3FKVZgTNO5D3UPhQlfpZHLG6obIN41y50kQuQ7pfkl1B_hGurVWuu83qy11GarUVO1WqWuwA3UG83zVqXRatbFp6Wpzwp8kLx63myq9YZa1Spaq66pjee_xR073w?type=png)](https://mermaid.live/edit#pako:eNqNlftu2jAUxl_FcqWJdqEihNsidRKBtnSjgEomoZEqMolDMoKdJY6AXv7dA-wR9ySzHS4Ja7tGCMXh-53z-eT48Agd6mKoQy-kK8dHMQNm1yKAX0k6m8co8sENiVJmYpLQOJlaUC7Bdm3B-0wtrgn_dQ2mM8QcXwEJ_mmHmCggEICdBA_4viA3uZwFS5wcIwcZJq5FjuyMYvoDOyygZG_m8KjoB5TLn8HobviFK_sBwSguHcyAP79-Az9wXUzk-rTACkriPc76_24qB75tt80YJsKaEVJnwYPtHwjbDk6SgMwLqXsyr3nXHoxVUaIYkSSiCS5VFLVoMtNkNtumqKdvI8bAdG9zaztzWyy_ICSakbdpyIIeRu7eX0G9E8uU1YItVam8YKsq5YPh3a3YRB9tcDyg8fLNWvXNzq5K_Pa1-siYMjoX9YfDEZdf0Rhg5PhANhTDEWB6AdpKJdYZDjptsWcHsdJ0bTNeIJvdK8ANlhfloxpnYskZBmcM5CxmlGAwwGxF40VBbBhSOB71b0T8cRQGDDAKNBAh5hdPixRJ-ZMFTW5bRLTgE7gSqGczcAGSYL6kgVsSu7IJZqevRxgzxDC4FgGuxUGc22sRQDwVpD3_H9oTaG8im6iI-gX0ygQfQMZet81LkUrgB7Nl4f0MHJkVWg5eT_hXLzuZRv9y0JX5CF7xADLOGRDOP4KSWhbrU_6A-3mzbYYp44d61znZ6rXmkTmzl2S2O1-5nu_TWZSopJKsB45aQCr37Vx9sZ13jZm1_fCbWRw7uXEh585hDBVTbcFdEB5g876BmquLE6Ik6WIPMDmjOzTkh8MLwlA_8T55SsJiusD6iaZp2_vyKnCZr1ej9VEAGuXh2ew9cC4EmCg9RYwZhW8l7yavEdtVhMYwFNEjSvaGtqmhApc4XqLA5f9Rj4Ljk8fHS2xBnd-62EN8blnQIs9cilJGxxviQJ3FKVZgTNO5D3UPhQlfpZHLG6obIN41y50kQuQ7pfkl1B_hGurVWuu83qy11GarUVO1WqWuwA3UG83zVqXRatbFp6Wpzwp8kLx63myq9YZa1Spaq66pjee_xR073w)


## LiquidTimeConstant (LTC) Class


### Core components
- Backbone network: Takes combined input+hidden state, processes through 2 linear layers with tanh
- Three specialized networks:
  * time_net: Controls temporal dynamics
  * state_net_g: Transforms state for short-term memory
  * state_net_h: Transforms state for long-term memory
- Learnable parameters:
  * tau: Time constants (initialized to ones)
  * A: Bias terms (initialized randomly)

### Forward pass:
1. Combines input and hidden state
2. Processes through backbone network
3. Computes time-constant factor using sigmoid
4. Applies state transformations
5. Uses time-dependent gating to blend states

# LTC diagram with explanations
[![](https://mermaid.ink/img/pako:eNqNV21P4zgQ_itWTiskRLlCeWmj00rAHuzqYA8dlU667apyk2lj1bGD7VC6lP9-45e8tKW78KFK4pnx88w8MzYvUSJTiOJopmiRkeHlSIwEwb8PH8gXUZSG3CuZgNZMzPyCLife1i2Pm-Vvo2jLI_runZyjWXLY8iJTxnnM2SwzE17CgTZKziH-rdfrhefOgqUmi4-K5wqb_WMujobHetsHeCxBJPDHRP3-cX__34waMpOg0TTe33dfO2TIciAaFMPvKTU0fPa-hlFOcqC6VJDjq65XhZaKKKApItZhg4eMFoCRybcJNUl2gGEfxxzEAZkCNRhCf1_j3zwZBDFmYioRuwP0BZ9VTg2Too2eGZJIYSgTep2BNjQvdJsSEwbUE-U1ZDTgNr2KGvgV4gYniLQlgKGiQltkoMg9NdmGAFrLY7ts2Wx4xPYXkQnyDyRyJpil-IYqNkO1VFEwMX-3KvKMIoq7khvW-YzlIhe4u3gjrykKo8npNROpxvpyVwKdscKqxqanUVSHuLAFot2D5wKU0XuESzknGLElpT9pkpHM7q0LSFBR7IfTICl8Jnwx3M8tmwPJ6JOtU14Fp4LypQ7as_HtKm6xlKVy-xDNrDEVIEvNlzs0JjCZR5iLW7rEWny18kIkbYndl6qQGposPBg6YR4uB6oEbhxW_gLAjKC8Sk8lySCZh7XPwHHNKFSpRbqQak4mYKluEJ1jEMeFv9FkUhCTYWdS254J5bBDkrfDq7ekiJ8rCVYWsWuMzsWCKvj5SKqcW6KbKcBGfq_qJjSZT6QA3P0yPJKvYGwqfiq7K4nQwrAg8Iw5TOr6dMitrQGmOy-kwmqbdQF1mp7Rfhg2ImxyjmLJqTKWGbpiihGGq60PssD3vfYWWNtaZzuENUMNuWHfSu-N-7ZLV1c4w5TkFmY95siUy0VYv6ScYpdpJKgN6j8lBQ5PlEZYv0hpYVBi0g1OLw-9gyhq9Ak8z7mQi4omCOteaiApm05BgeNao9lBNYdcqiVS_bQUNGcJuXMfAs_rMOUbonfYA25ao7Bw7BL5ZHOOmLeI-IIlGRUzqOp5iZMYp1AD0Hq-STaMDErSgCtvcDnqZZHasU8WDFUtYOG47uiov0tjsdyAAOWTsd5afn3crGM-tn22u2rLr9VeS-BY_nf3VyLzCXPtFXKOkrJf1qaZ7_BWMUDNnKZc03QmVENan8y1Ml3gkCib7w51grb52lRnUxhqJ7vZrMpEYVFsXYycAQ4zhVG0ZesNp0rmrRjhCNmhPOmSZwkjSx5y2Z4l2KUEd2noBv7IcePUGjassK1S5kZMlYD7teys4f3lLcdjHGs8MHZdILDzBfgN_bf61kY6nY-ra7xUhfqEY4_plT3DdxkbsJMKE1LU83xVT1_vVN-tnNNNyVIsXe2X4omsLZxVmGPeCbd05vUJWAtl5Q9Sb1bt5PEEC4JTJARbD-qfnW2YgGEshO5chbb11m4bZ1zdmBoIQf8Bq3NylsOK1g7T8OIz4ftwTQSrUMORiA4ivHrllKX4r8CL9R5FKOEcj-EYH1OYUrxyjKKReEVTWhr5sBRJFBuFV_ZIyXKWRfEUr5_45ul9YhTnR15_Laj4T8q8csHXKH6JnqP4uHd8eNwbDHqD825_cHYyOIiWUTzoHp6fdbtH5-dHJyenJ_3j14Poh_M_OhycDs77Z8eD0_6g2-_2X_8HbONX3w?type=png)](https://mermaid.live/edit#pako:eNqNV21P4zgQ_itWTiskRLlCeWmj00rAHuzqYA8dlU667apyk2lj1bGD7VC6lP9-45e8tKW78KFK4pnx88w8MzYvUSJTiOJopmiRkeHlSIwEwb8PH8gXUZSG3CuZgNZMzPyCLife1i2Pm-Vvo2jLI_runZyjWXLY8iJTxnnM2SwzE17CgTZKziH-rdfrhefOgqUmi4-K5wqb_WMujobHetsHeCxBJPDHRP3-cX__34waMpOg0TTe33dfO2TIciAaFMPvKTU0fPa-hlFOcqC6VJDjq65XhZaKKKApItZhg4eMFoCRybcJNUl2gGEfxxzEAZkCNRhCf1_j3zwZBDFmYioRuwP0BZ9VTg2Too2eGZJIYSgTep2BNjQvdJsSEwbUE-U1ZDTgNr2KGvgV4gYniLQlgKGiQltkoMg9NdmGAFrLY7ts2Wx4xPYXkQnyDyRyJpil-IYqNkO1VFEwMX-3KvKMIoq7khvW-YzlIhe4u3gjrykKo8npNROpxvpyVwKdscKqxqanUVSHuLAFot2D5wKU0XuESzknGLElpT9pkpHM7q0LSFBR7IfTICl8Jnwx3M8tmwPJ6JOtU14Fp4LypQ7as_HtKm6xlKVy-xDNrDEVIEvNlzs0JjCZR5iLW7rEWny18kIkbYndl6qQGposPBg6YR4uB6oEbhxW_gLAjKC8Sk8lySCZh7XPwHHNKFSpRbqQak4mYKluEJ1jEMeFv9FkUhCTYWdS254J5bBDkrfDq7ekiJ8rCVYWsWuMzsWCKvj5SKqcW6KbKcBGfq_qJjSZT6QA3P0yPJKvYGwqfiq7K4nQwrAg8Iw5TOr6dMitrQGmOy-kwmqbdQF1mp7Rfhg2ImxyjmLJqTKWGbpiihGGq60PssD3vfYWWNtaZzuENUMNuWHfSu-N-7ZLV1c4w5TkFmY95siUy0VYv6ScYpdpJKgN6j8lBQ5PlEZYv0hpYVBi0g1OLw-9gyhq9Ak8z7mQi4omCOteaiApm05BgeNao9lBNYdcqiVS_bQUNGcJuXMfAs_rMOUbonfYA25ao7Bw7BL5ZHOOmLeI-IIlGRUzqOp5iZMYp1AD0Hq-STaMDErSgCtvcDnqZZHasU8WDFUtYOG47uiov0tjsdyAAOWTsd5afn3crGM-tn22u2rLr9VeS-BY_nf3VyLzCXPtFXKOkrJf1qaZ7_BWMUDNnKZc03QmVENan8y1Ml3gkCib7w51grb52lRnUxhqJ7vZrMpEYVFsXYycAQ4zhVG0ZesNp0rmrRjhCNmhPOmSZwkjSx5y2Z4l2KUEd2noBv7IcePUGjassK1S5kZMlYD7teys4f3lLcdjHGs8MHZdILDzBfgN_bf61kY6nY-ra7xUhfqEY4_plT3DdxkbsJMKE1LU83xVT1_vVN-tnNNNyVIsXe2X4omsLZxVmGPeCbd05vUJWAtl5Q9Sb1bt5PEEC4JTJARbD-qfnW2YgGEshO5chbb11m4bZ1zdmBoIQf8Bq3NylsOK1g7T8OIz4ftwTQSrUMORiA4ivHrllKX4r8CL9R5FKOEcj-EYH1OYUrxyjKKReEVTWhr5sBRJFBuFV_ZIyXKWRfEUr5_45ul9YhTnR15_Laj4T8q8csHXKH6JnqP4uHd8eNwbDHqD825_cHYyOIiWUTzoHp6fdbtH5-dHJyenJ_3j14Poh_M_OhycDs77Z8eD0_6g2-_2X_8HbONX3w)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple

class LiquidTimeConstant(nn.Module):
    """
    Closed-form continuous-time cell with a learnable per-neuron time constant
    and equilibrium offset, following Hasani et al. (2022) CfC-style dynamics.

    Architectural role in SolaceCore:
        Each "block" actor in the SolaceCore actor graph contains an LTC cell
        (or a small stack of them). Once a cube has been mentored long enough
        for its LTC to absorb the transformer's behavior, ownership of that
        cube flips from TRANSFORMER to LNN_OWNED and the multi-dim attention
        vector for that cube collapses into the dynamic state of one cell.
        That is the vector-to-neuron transition.

    Math:
        Combined input z_t = [x_t ; h_{t-1}]
        features  = backbone(z_t)
        f_t       = sigmoid(time_net(features))            (per-neuron rate)
        g_x       = state_net_g(features)                  (short-term branch)
        h_x       = state_net_h(features)                  (long-term branch)
        gate      = sigmoid(-(f_t / tau_pos) * t)          (time-gated blender)
        h_new     = gate * g_x + (1 - gate) * (h_x + A)

    Where:
        tau is a learnable per-neuron time constant, kept positive via
        softplus so the bounded-dynamics theorem (Hasani 2020,
        tau_i / (1 + tau_i * W_i) <= tau_sys_i <= tau_i) holds in spirit.
        A is the input-driven equilibrium offset present in the long-term
        branch. Both were declared but unused in the Kaggle prototype;
        wiring them in is the first correction this north-star applies.
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Backbone: combined input -> features.
        self.backbone = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
        )

        # Temporal-rate and state-transformation heads.
        self.time_net = nn.Linear(hidden_size, hidden_size)
        self.state_net_g = nn.Linear(hidden_size, hidden_size)
        self.state_net_h = nn.Linear(hidden_size, hidden_size)

        # Per-neuron time constant. We initialize tau via softplus^{-1}(1.0)
        # so tau_pos starts at ~1.0 and is free to grow or shrink under the
        # softplus floor.
        self.tau_raw = nn.Parameter(torch.full((hidden_size,),
                                                float(np.log(np.e - 1.0))))
        # Equilibrium offset of the long-term branch.
        self.A = nn.Parameter(torch.zeros(hidden_size))

    @property
    def tau(self) -> torch.Tensor:
        """Positive per-neuron time constants."""
        return F.softplus(self.tau_raw) + 1e-6

    def forward(self,
                x: torch.Tensor,   # [batch, input_size]
                h: torch.Tensor,   # [batch, hidden_size]
                t: torch.Tensor,   # [batch] or scalar; positional or wall-clock
                ) -> Tuple[torch.Tensor, torch.Tensor]:
        combined = torch.cat([x, h], dim=-1)
        features = self.backbone(combined)

        f_t = torch.sigmoid(self.time_net(features))   # [batch, hidden]
        g_x = self.state_net_g(features)
        h_x = self.state_net_h(features)

        # Per-neuron time scaling: a larger tau means a slower transition
        # from short-term to long-term. The gate argument is -f_t * t / tau.
        if t.dim() == 1:
            t_b = t.view(-1, 1)
        else:
            t_b = t

        gate = torch.sigmoid(-(f_t / self.tau) * t_b)

        # Equilibrium offset A enters the long-term branch.
        h_new = gate * g_x + (1.0 - gate) * (h_x + self.A)
        return h_new, h_new


## TransformerLNN Class

### Core components
- input_proj: Projects input to higher dimension
- attention: Multi-head attention for parallel processing
- ltc: Liquid time-constant layer
- output_proj: Projects back to input dimension
- Two layer normalizations for stability

### Forward pass:
1. Projects input to higher dimension
2. Applies self-attention and normalizes
3. Processes sequence through LTC:
   - Maintains a running state
   - Updates state for each time step
   - Collects outputs
4. Combines and normalizes outputs
5. Projects back to original dimension

# Tensor operations with explanations
[![](https://mermaid.ink/img/pako:eNqNVm1v2zYQ_iuEim5rYGf1W5IKQ4GmSdoCyRYsAQasLgJaPFmsKVIlqdlunP--44tsxXFe9Iki73jH53nuyNskUwySNJlqWhXk-ngsx5Lg9_o1-SKr2pITammYMvUkWPmFG7fwdZy0rJJvwdAb26WAliXJuRCp4NPCTkQNHWO1mkH6ajAYxHF3zpkt0l61aHJwH_c7LNaBruBHDTKDPyb69_d7e1cFrSDd2yNfJ9RmRYcY-HEjQHaio-E_4Vu0PV3QshLe-pqXgKaagyFsO_Xt4BaDe4cvMle6pJYr-Uz4R0Iai79mHQwkuwc3t5wKcqlVBsZwOX0Auze42Rh4VB547aJh27NFx1TD8sV0VFp933CB4JIP7HttbAnS-jN3yQWtTECOWEUKzhhIwvD00jTAdck5nwHRgPS4ZKgkvKRTeISGsAeGjfkDC8J6VgPBsSWCLvmbZwVoDF1hdEzas0lUHjJ-hJprTaVx3KPnJbXFFjGt5Ru37PSy5ZGSc6BaGvdjAQc7WNrepsXREoRQ8xez5GJIdzDM5KIWlnc_A2XkQzMdgfunoJZwS5gCg_hFhM64ZEhgWSltqbQIlfAgmYJXpk1fQf9z5JUuAIqcwKICbQ0RSs0wBV9Y0f6UZgUpXA65ymqDVaecJvIcNGZEqm1Q7sN_fv1xF-w43cDdWKTkM5VM4P6u3HZg3Pjc1z9q5aXQTmg2mygJGPQ4DsmfYOdKzyKql7WulIENoKcLq2mGwMxgSXKgtkbptYGkhAFURDiFxNpEOBqLoJsNH9UuBW1GU2ohNqzuMXW18gnpk9MnOf-oJJ5ZuLpdtziSo-Ti-glknCGqc_T-tZ2MXJvEk5iSahtPgGC5-LiV3WajBShWKsOEzwIw5Nj9b_KNJd9OtZxwibmYAnMgv6Dc5LSL0UpSQqn0smlDfOGa-1pj1rf8jIo19sdUULxJUIuCYQtiRMK8DcAjasT6wH77V21dv7ivSL90E5bckdqWD7XYtg56fJW_y1-sRLUrTEStAWkDm9cDnVNEuJEg-Q2r4U1cj32pGyjbmLSa0pvtfntFEVJqnmydqCwJme8fYS5e6KTbfb86A4wVrgrXLFb-eglmbuRtogCwcehG-giWl8hq0-iecPLEF64teJemgNvZWO_k7_h2AUxrL3qG4ndXl1n54gqOzTbxHBEu9Bb-eeIsfbjnPFyGJV-E1ITHrqlib7iuzGDkjuhOds_YD8O2Xgp4tTEeMF9FmQTDNVwBpJ2UbzySToKsl5QzfB_euh3GiS3AVXKKQwY5xdY_TsbyDk1pbdXVUmZJajW-7xKt6mmRpDkVBv_qCgmGE06xUMr1bEXlv0qVjQv-JultskjS7sH-6PBo0Bv1-4PRQa-PpZAscXpwsP-uf9Tr9UbD0VG_1xse3HWSn36L_n6_d_h2OByMBsOjUe_t4bCTIAxW6YvwwvUP3bv_Ae79qTM?type=png)](https://mermaid.live/edit#pako:eNqNVm1v2zYQ_iuEim5rYGf1W5IKQ4GmSdoCyRYsAQasLgJaPFmsKVIlqdlunP--44tsxXFe9Iki73jH53nuyNskUwySNJlqWhXk-ngsx5Lg9_o1-SKr2pITammYMvUkWPmFG7fwdZy0rJJvwdAb26WAliXJuRCp4NPCTkQNHWO1mkH6ajAYxHF3zpkt0l61aHJwH_c7LNaBruBHDTKDPyb69_d7e1cFrSDd2yNfJ9RmRYcY-HEjQHaio-E_4Vu0PV3QshLe-pqXgKaagyFsO_Xt4BaDe4cvMle6pJYr-Uz4R0Iai79mHQwkuwc3t5wKcqlVBsZwOX0Auze42Rh4VB547aJh27NFx1TD8sV0VFp933CB4JIP7HttbAnS-jN3yQWtTECOWEUKzhhIwvD00jTAdck5nwHRgPS4ZKgkvKRTeISGsAeGjfkDC8J6VgPBsSWCLvmbZwVoDF1hdEzas0lUHjJ-hJprTaVx3KPnJbXFFjGt5Ru37PSy5ZGSc6BaGvdjAQc7WNrepsXREoRQ8xez5GJIdzDM5KIWlnc_A2XkQzMdgfunoJZwS5gCg_hFhM64ZEhgWSltqbQIlfAgmYJXpk1fQf9z5JUuAIqcwKICbQ0RSs0wBV9Y0f6UZgUpXA65ymqDVaecJvIcNGZEqm1Q7sN_fv1xF-w43cDdWKTkM5VM4P6u3HZg3Pjc1z9q5aXQTmg2mygJGPQ4DsmfYOdKzyKql7WulIENoKcLq2mGwMxgSXKgtkbptYGkhAFURDiFxNpEOBqLoJsNH9UuBW1GU2ohNqzuMXW18gnpk9MnOf-oJJ5ZuLpdtziSo-Ti-glknCGqc_T-tZ2MXJvEk5iSahtPgGC5-LiV3WajBShWKsOEzwIw5Nj9b_KNJd9OtZxwibmYAnMgv6Dc5LSL0UpSQqn0smlDfOGa-1pj1rf8jIo19sdUULxJUIuCYQtiRMK8DcAjasT6wH77V21dv7ivSL90E5bckdqWD7XYtg56fJW_y1-sRLUrTEStAWkDm9cDnVNEuJEg-Q2r4U1cj32pGyjbmLSa0pvtfntFEVJqnmydqCwJme8fYS5e6KTbfb86A4wVrgrXLFb-eglmbuRtogCwcehG-giWl8hq0-iecPLEF64teJemgNvZWO_k7_h2AUxrL3qG4ndXl1n54gqOzTbxHBEu9Bb-eeIsfbjnPFyGJV-E1ITHrqlib7iuzGDkjuhOds_YD8O2Xgp4tTEeMF9FmQTDNVwBpJ2UbzySToKsl5QzfB_euh3GiS3AVXKKQwY5xdY_TsbyDk1pbdXVUmZJajW-7xKt6mmRpDkVBv_qCgmGE06xUMr1bEXlv0qVjQv-JultskjS7sH-6PBo0Bv1-4PRQa-PpZAscXpwsP-uf9Tr9UbD0VG_1xse3HWSn36L_n6_d_h2OByMBsOjUe_t4bCTIAxW6YvwwvUP3bv_Ae79qTM)

In [ ]:
class TransformerLNN(nn.Module):
    """
    Hybrid architecture combining Transformer attention (for pattern recognition)
    with LTC dynamics (for temporal processing)
    """
    def __init__(self, 
                 input_size: int,  # Size of each input feature
                 hidden_size: int, # Size of internal representations
                 num_heads: int = 4,  # Number of attention "experts"
                 dropout: float = 0.1):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        
        # INPUT PROCESSING
        # Projects input to higher dimension for richer representation
        # Like resizing data to give the network more room to work
        self.input_proj = nn.Linear(input_size, hidden_size)
        
        # TRANSFORMER PATH: Pattern Recognition
        # Multi-head attention: Multiple "experts" look at data simultaneously
        # Each head can specialize in finding different types of patterns
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout)
        
        # LTC PATH: Time-Aware Processing
        # Processes temporal aspects and maintains dynamic memory
        self.ltc = LiquidTimeConstant(hidden_size, hidden_size)
        
        # OUTPUT PROCESSING
        # Projects back to original dimension for predictions
        self.output_proj = nn.Linear(hidden_size, input_size)
        
        # STABILIZATION
        # Keep values in check during processing (like keeping measurements on same scale)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, 
                x: torch.Tensor,  # Input shape: [batch, seq_len, input_size]
                times: Optional[torch.Tensor] = None,  # Time info: [batch, seq_len]
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass combining parallel pattern recognition (Transformer)
        with sequential time-aware processing (LTC)
        """
        batch_size, seq_len, _ = x.shape
        
        # Generate time information if none provided
        # Like adding timestamps if they're missing
        if times is None:
            times = torch.arange(seq_len, dtype=torch.float32, device=x.device)
            times = times.unsqueeze(0).expand(batch_size, -1)
        
        # PROJECT INPUT
        # Transform to richer representation
        h = self.input_proj(x)
        
        # TRANSFORMER PATH: Find Patterns
        # Rearrange for attention ([seq_len, batch, hidden])
        h_att = h.transpose(0, 1)
        # Let multiple experts look at sequence relationships
        h_att, _ = self.attention(h_att, h_att, h_att, attn_mask=mask)
        # Rearrange back ([batch, seq, hidden])
        h_att = h_att.transpose(0, 1)
        # Stabilize with residual connection and normalization
        h_att = self.norm1(h + h_att)
        
        # LTC PATH: Process Time Information
        # Initialize memory state
        ltc_state = torch.zeros(batch_size, self.hidden_size, device=x.device)
        outputs = []
        
        # Process sequence step by step, maintaining memory
        for t in range(seq_len):
            ltc_in = h_att[:, t]  # Current timestep
            # Update memory and get output
            ltc_out, ltc_state = self.ltc(ltc_in, ltc_state, times[:, t])
            outputs.append(ltc_out)
        
        # COMBINE FEATURES
        # Stack timestep outputs
        outputs = torch.stack(outputs, dim=1)
        # Merge time-aware features with pattern features
        outputs = self.norm2(outputs + h_att)
        
        # GENERATE PREDICTIONS
        # Project back to input size
        y = self.output_proj(outputs)
        
        return y  # Shape: [batch, seq_len, input_size]
    def get_attention_weights(self, x: torch.Tensor) -> torch.Tensor:
        """Helper method to extract attention weights for visualization"""
        h = self.input_proj(x)
        h = h.transpose(0, 1)
        _, attn_weights = self.attention(h, h, h, need_weights=True)
        return attn_weights

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import seaborn as sns
from typing import List, Dict, Optional
from datetime import datetime

class TrainingVisualizer:

    def __init__(self, save_intervals: int = 10, plot_intervals: int = 5):
        self.save_intervals = save_intervals
        self.plot_intervals = plot_intervals

        # History tracking
        self.losses: List[float] = []
        self.accuracies: List[float] = []
        self.ltc_states: List[torch.Tensor] = []
        self.attn_weights: List[torch.Tensor] = []
        self.timestamps: List[str] = []

    def update_metrics(self, loss: float, accuracy: float, 
                      ltc_state: torch.Tensor, attn_weights: torch.Tensor):

        """Update training metrics"""

        self.losses.append(loss)
        self.accuracies.append(accuracy)
        self.ltc_states.append(ltc_state.detach().cpu())
        self.attn_weights.append(attn_weights.detach().cpu())
        self.timestamps.append(datetime.now().strftime("%H:%M:%S"))

    def plot_training_progress(self, epoch: int, batch_idx: int, x: torch.Tensor, y: torch.Tensor, y_pred: torch.Tensor):
        """Plot comprehensive training visualizations"""
        # Convert tensors to 2D by taking first channel/feature
        x_plot = x[:, :, 0].cpu().detach().numpy()  # Take first feature dimension
        y_plot = y[:, :, 0].cpu().detach().numpy()
        y_pred_plot = y_pred[:, :, 0].cpu().detach().numpy()
     
        clear_output(wait=True)
        plt.figure(figsize=(20, 12))
      
        # Plot actual visualizations using 2D data
        plt.subplot(231)
        plt.plot(self.losses[-100:], label='Training Loss', color='blue')
        plt.title(f'Loss Curve (Current: {self.losses[-1]:.4f})')
        plt.xlabel('Last 100 Steps')
        plt.ylabel('Loss')
        plt.grid(True)

        # Use the 2D versions for prediction plots
        plt.subplot(235)
        idx = 0  # Plot first sequence in batch
        plt.plot(y_plot[idx], label='Ground Truth', color='blue', alpha=0.7)
        plt.plot(y_pred_plot[idx], label='Prediction', color='red', alpha=0.7)
        plt.title('Prediction vs Ground Truth')
        plt.xlabel('Sequence Step')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)

    def generate_synthetic_data(num_samples: int = 1000,
                            seq_length: int = 50,
                            input_dim: int = 10,
                            device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
                            ) -> tuple[torch.Tensor, torch.Tensor]:

        """Generate synthetic sequence data with multiple patterns"""

        # Time steps

        t = torch.linspace(0, 10, seq_length, device=device)
        t = t.view(1, -1, 1).repeat(num_samples, 1, input_dim)     

        # Generate diverse patterns

        patterns = []

        for i in range(input_dim):
 
            # Combine sine waves with different frequencies

            freq1 = (i + 1) * 0.5
            freq2 = (i + 1) * 0.25

            pattern = torch.sin(2 * np.pi * freq1 * t[..., i]) + \
                    0.5 * torch.sin(2 * np.pi * freq2 * t[..., i])

            patterns.append(pattern)      

        # Combine patterns

        x = torch.stack(patterns, dim=-1)

        # Create target with transformations

        y = torch.roll(x, shifts=-1, dims=1) * 1.5 + 0.5
        return x, y

    
    def train_epoch(model: nn.Module,
                    train_loader: torch.utils.data.DataLoader,
                    criterion: nn.Module,
                    optimizer: torch.optim.Optimizer,
                    device: str,
                    visualizer,
                    epoch: int) -> float:

        """Train for one epoch"""

        model.train()
        total_loss = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()           

            # Forward pass

            y_pred = model(x)
            loss = criterion(y_pred, y)       

            # Backward pass

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Update visualizations

            if batch_idx % visualizer.save_intervals == 0:
                with torch.no_grad():
                    
                    # Get model states

                    ltc_state, _ = model.ltc(
                        model.input_proj(x[0:1]), 
                        torch.zeros(1, model.hidden_size, device=device),
                        torch.zeros(1, device=device)
                    )

                    attn_weights = model.get_attention_weights(x[0:1])

                    # Compute accuracy

                    mse = ((y_pred - y) ** 2).mean().item()
                    accuracy = 100 * (1 - min(mse, 1))                  

                    # Update visualizer

                    visualizer.update_metrics(loss.item(), accuracy, ltc_state, attn_weights)
     
            # Plot progress

            if batch_idx % visualizer.plot_intervals == 0:
                visualizer.plot_training_progress(epoch, batch_idx, x, y, y_pred)

        return total_loss / len(train_loader)

## 1. Setup Training Parameters

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


# Model parameters

INPUT_SIZE = 10
HIDDEN_SIZE = 64
NUM_HEADS = 4



# Training parameters

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 50


# Data parameters

NUM_SAMPLES = 1000
SEQ_LENGTH = 50

# Device configuration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

## 2. Generate Synthetic Data

In [ ]:
# Generate training data

x_train, y_train = TrainingVisualizer.generate_synthetic_data(

    num_samples=NUM_SAMPLES,
    seq_length=SEQ_LENGTH,
    input_dim=INPUT_SIZE,
    device=device

)



# Create dataloader

train_dataset = torch.utils.data.TensorDataset(x_train, y_train)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)



# Plot sample sequence

plt.figure(figsize=(12, 4))
plt.plot(x_train[0, :, 0].cpu().numpy(), label='Input')
plt.plot(y_train[0, :, 0].cpu().numpy(), label='Target')
plt.title('Sample Sequence')
plt.legend()
plt.grid(True)
plt.show()

## 3. Create and Initialize Model

In [ ]:
# Initialize model

model = TransformerLNN(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS

).to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Initialize visualizer

visualizer = TrainingVisualizer()

## 4. Training Loop with Visualization

In [ ]:
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    

    for batch_idx, (x, y) in enumerate(train_loader):
        
        # Forward pass
        y_pred = model(x)
        loss = criterion(y_pred, y)
    
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Update visualization
        if batch_idx % visualizer.save_intervals == 0:
            
            with torch.no_grad():
                
                # Get model states
                batch_input = x[0:1]  # Shape: [1, seq_len, input_dim]
                projected = model.input_proj(batch_input)  # Shape: [1, seq_len, hidden_size]

                # Take first timestep for LTC state
                ltc_input = projected[:, 0, :]  # Shape: [1, hidden_size]
                ltc_state, _ = model.ltc(
                    ltc_input,
                    torch.zeros(1, HIDDEN_SIZE, device=device),
                    torch.zeros(1, device=device)
                )

        attn_weights = model.get_attention_weights(batch_input)


        # Compute accuracy

        mse = ((y_pred - y) ** 2).mean().item()

        accuracy = 100 * (1 - min(mse, 1))

        

        # Update visualizer

        visualizer.update_metrics(loss.item(), accuracy, ltc_state, attn_weights)

        

        # Plot progress

        if batch_idx % visualizer.plot_intervals == 0:

            visualizer.plot_training_progress(epoch, batch_idx, x, y, y_pred)

    

    # End of epoch summary

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Average Loss: {avg_loss:.4f}")

## 5. Final Evaluation

In [ ]:
# Generate test data

x_test, y_test = TrainingVisualizer.generate_synthetic_data(

    num_samples=10,

    seq_length=SEQ_LENGTH,

    input_dim=INPUT_SIZE,

    device=device

)



# Model evaluation

model.eval()

with torch.no_grad():

    y_pred = model(x_test)

    test_loss = criterion(y_pred, y_test)

    

    # Plot predictions

    plt.figure(figsize=(15, 5))

    for i in range(3):  # Plot first 3 sequences

        plt.subplot(1, 3, i+1)

        plt.plot(y_test[i, :, 0].cpu().numpy(), label='Ground Truth', alpha=0.7)

        plt.plot(y_pred[i, :, 0].cpu().numpy(), label='Prediction', alpha=0.7)

        plt.title(f'Sequence {i+1}')

        plt.legend()

        plt.grid(True)

    plt.tight_layout()

    plt.show()

    

print(f"Final Test Loss: {test_loss:.4f}")

## 6. Analyze Learning Progression

In [ ]:
# Plot training history

plt.figure(figsize=(15, 5))



# Loss progression

plt.subplot(131)

plt.plot(visualizer.losses, label='Training Loss')

plt.title('Loss Progression')

plt.xlabel('Step')

plt.ylabel('Loss')

plt.grid(True)



# Accuracy progression

plt.subplot(132)

plt.plot(visualizer.accuracies, label='Accuracy')

plt.title('Accuracy Progression')

plt.xlabel('Step')

plt.ylabel('Accuracy (%)')

plt.grid(True)



# Final attention pattern

plt.subplot(133)

# Extract a 2D slice from the 3D tensor

attn_weights_2d = visualizer.attn_weights[-1][0].cpu().numpy()



# Plot the 2D attention weights

sns.heatmap(attn_weights_2d, cmap='viridis')

plt.title('Final Attention Pattern')



plt.tight_layout()

plt.show()